# 02. Базовое решение

По требованиям базовое решение это «простая модель или правило, с которым сравниваются остальные». Метрика без такого сравнения ничего не говорит, поэтому здесь два уровня:

- **A. Нижние границы:** простые правила для GSM8K (без модели) и случайный выбор для тестов с вариантами ответа. Показывают, что такое «ничего не умеем».
- **B. Исходная модель без дообучения:** zero-shot и 5-shot. Это настоящая точка отсчёта для всех методов дообучения.

Плюс разбираем, **почему strict-match и flexible-extract различаются у исходной модели**, и выбираем основную метрику.

In [ ]:
import os, sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# корень репозитория: ноутбук можно запускать и из notebooks/, и из корня
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

from datasets import load_dataset
from src.config import load_config

cfg = load_config(["configs/base.yaml"])
MODEL = cfg["model"]["name"]
BASE_DIR = ROOT / "results" / ("baseline_" + MODEL.split("/")[-1])
print("модель:", MODEL, "| результаты:", BASE_DIR)

## A. Правила без модели (GSM8K test)

Правила намеренно примитивные: берём числа из условия и применяем одну операцию. Ответ засчитывается, если число совпадает с эталонным. Для сравнения добавлен «константный» ответ (самый частый ответ из train).
Доверительный интервал считаем по Вильсону: на 1319 задачах он заметно шире, чем кажется.

In [ ]:
test = load_dataset("openai/gsm8k", "main", split="test")
train = load_dataset("openai/gsm8k", "main", split="train")

NUM = re.compile(r"\d[\d,]*(?:\.\d+)?")

def nums(text):
    out = []
    for m in NUM.findall(text):
        try:
            out.append(float(m.replace(",", "")))
        except ValueError:
            pass
    return out

def final(sol):
    return float(sol.split("####")[-1].strip().replace(",", ""))

gold = np.array([final(a) for a in test["answer"]])
train_answers = pd.Series([final(a) for a in train["answer"]])
mode_answer = float(train_answers.mode().iloc[0])

rules = {
    "последнее число в условии": lambda q: (nums(q) or [None])[-1],
    "первое число в условии": lambda q: (nums(q) or [None])[0],
    "сумма всех чисел в условии": lambda q: sum(nums(q)) if nums(q) else None,
    "произведение двух последних чисел": lambda q: (nums(q)[-1] * nums(q)[-2]) if len(nums(q)) >= 2 else None,
    f"константа {mode_answer:g} (мода train)": lambda q: mode_answer,
}

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

rows = []
for name, f in rules.items():
    preds = [f(q) for q in test["question"]]
    ok = np.array([p is not None and abs(p - g) < 1e-6 for p, g in zip(preds, gold)])
    lo, hi = wilson(ok.sum(), len(ok))
    rows.append({"правило": name, "accuracy, %": 100 * ok.mean(), "95% ДИ, %": f"{100*lo:.1f}–{100*hi:.1f}"})

rule_df = pd.DataFrame(rows).set_index("правило").round(2)
rule_df

Для тестов с вариантами ответа нижняя граница это случайный выбор. Это приближение: у ARC и MMLU бывают вопросы с другим числом вариантов, но почти все четырёхвариантные.

| Бенчмарк | Случайный выбор |
|---|---|
| ARC-Easy / ARC-Challenge | ≈25% |
| HellaSwag | 25% (4 окончания) |
| MMLU | ≈25% |

## B. Исходная модель

Прогоните оценку один раз (дольше всего считается GSM8K 5-shot и MMLU). Либо в терминале `bash scripts/run.sh baseline`, либо флагом ниже. Если `eval.json` уже есть, повторный запуск пропускается.

In [ ]:
import subprocess

RUN_BASELINE_EVAL = False   # True: запустить оценку исходной модели прямо из ноутбука (нужна GPU)

if RUN_BASELINE_EVAL and not (BASE_DIR / "eval.json").exists():
    subprocess.run([sys.executable, "-m", "src.eval", "--baseline", "--config", "configs/base.yaml",
                    "--eval_config", "configs/eval_baseline.yaml"], check=True)

assert (BASE_DIR / "eval.json").exists(), "Нет results/baseline_*/eval.json: запустите bash scripts/run.sh baseline"
ev = json.loads((BASE_DIR / "eval.json").read_text())

rows = []
for key, res in ev.items():
    task = key.split("|")[0]
    r = res.get(task, {})
    for k, v in r.items():
        if not isinstance(v, (int, float)) or "stderr" in k:
            continue
        metric, _, flt = k.partition(",")
        se = r.get(f"{metric}_stderr,{flt}")
        rows.append({"задача|шоты": key, "метрика": k, "значение, %": 100 * v,
                     "ст. ошибка, п.п.": 100 * se if isinstance(se, (int, float)) else np.nan})
base_df = pd.DataFrame(rows).round(2)
base_df

### Почему у GSM8K две метрики

В lm-eval у GSM8K два способа извлечь ответ:
- **strict-match** ищет `#### N`. Это формат, которому дообучение учит модель. Исходная модель в режиме 0-shot этого формата не знает.
- **flexible-extract** берёт последнее число в ответе, формат не важен.

Если сравнить дообученную модель по strict-match с исходной по strict-match, часть «прироста» будет приростом **формата**, а не математики. Поэтому:
- **основная метрика во всей работе: flexible-extract, 0-shot**, одинаковая для всех методов;
- strict-match приводим как дополнительную: она показывает, насколько модель усвоила формат;
- для исходной модели честная точка отсчёта это **лучший из вариантов без обучения**, обычно 5-shot, который показывает формат в промпте.

In [ ]:
def get(key, metric):
    return ev.get(key, {}).get(key.split("|")[0], {}).get(metric)

variants = {
    "исходная, 0-shot, flexible": ("gsm8k|0shot", "exact_match,flexible-extract"),
    "исходная, 0-shot, strict":   ("gsm8k|0shot", "exact_match,strict-match"),
    "исходная, 5-shot, flexible": ("gsm8k|5shot", "exact_match,flexible-extract"),
    "исходная, 5-shot, strict":   ("gsm8k|5shot", "exact_match,strict-match"),
}
n_test = len(test)
bars = []
for label, (k, m) in variants.items():
    v = get(k, m)
    if v is None:
        continue
    lo, hi = wilson(round(v * n_test), n_test)
    bars.append((label, 100 * v, 100 * (v - lo), 100 * (hi - v), "#7f7f7f"))
for name, row in rule_df.iterrows():
    k = row["accuracy, %"] / 100
    lo, hi = wilson(round(k * n_test), n_test)
    bars.append((name, 100 * k, 100 * (k - lo), 100 * (hi - k), "#c9c9c9"))

bars = sorted(bars, key=lambda b: b[1])
fig, ax = plt.subplots(figsize=(8, 0.45 * len(bars) + 1.2))
ax.barh([b[0] for b in bars], [b[1] for b in bars], xerr=[[b[2] for b in bars], [b[3] for b in bars]],
        color=[b[4] for b in bars], capsize=3)
ax.set(xlabel="точность на GSM8K test, % (95% ДИ Вильсона)", title="GSM8K: правила и исходная модель")
plt.tight_layout()
plt.savefig(FIG / "baseline_gsm8k.png", bbox_inches="tight")
plt.show()

## C. Что именно отвечает исходная модель (опционально, нужна GPU)

Смотрим несколько ответов в 0-shot и считаем долю ответов, где модель вообще выдала `#### N`, и долю, где последнее число верное. Это объясняет разрыв strict и flexible и показывает типичные ошибки (не закончила решение, зациклилась, продолжила генерировать новые вопросы).

In [ ]:
RUN_SAMPLES = False   # True: сгенерировать ответы на N_SAMPLES задач (нужна GPU)
N_SAMPLES = 64

if RUN_SAMPLES:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODEL)
    tok.pad_token = tok.pad_token or tok.eos_token
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).cuda().eval()

    prompts = [cfg["data"]["prompt_template"].format(question=q) for q in test["question"][:N_SAMPLES]]
    outs = []
    for i in range(0, len(prompts), 16):
        enc = tok(prompts[i:i + 16], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    outs = [o.split("Question:")[0] for o in outs]   # как stop-последовательность в lm-eval

    has_marker = [bool(re.search(r"####\s*-?\d", o)) for o in outs]
    last_ok = [(nums(o)[-1:] == [g]) for o, g in zip(outs, gold[:N_SAMPLES])]
    print(f"есть '#### N': {np.mean(has_marker):.1%} | последнее число верное: {np.mean(last_ok):.1%}")
    for o, g in list(zip(outs, gold))[:3]:
        print("-" * 60, "\nэталон:", g, "\n", o.strip()[:500])

## D. Итог: что нужно превзойти

Сохраняем в `results/baseline_summary.json` ключевые числа, чтобы в 03 они подтягивались автоматически.

In [ ]:
summary = {
    "rules_gsm8k_pct": rule_df["accuracy, %"].to_dict(),
    "random_choice_pct": {"arc_easy": 25, "arc_challenge": 25, "hellaswag": 25, "mmlu": 25},
    "model": MODEL,
}
for label, (k, m) in variants.items():
    v = get(k, m)
    if v is not None:
        summary[label] = round(100 * v, 2)
(ROOT / "results" / "baseline_summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False))
summary

## Выводы (заполнить по результатам)

- Лучшее из правил даёт X% на GSM8K: это нижняя граница, любая модель должна быть заметно выше.
- Исходная модель: X% (0-shot, flexible), Y% (5-shot, flexible). Разрыв strict и flexible: Z п.п., причина формат `####`.
- Доверительный интервал на 1319 задачах около ±2.5 п.п. при точности ≈50%: выигрыши меньше этого без повторов по seed считаем неразличимыми.
- Общие навыки исходной модели (ARC, HellaSwag, MMLU) это ориентир для оценки забывания в 03.
- Возможное загрязнение: часть современных моделей видела GSM8K при претрейне, поэтому исходная точность может быть завышена. Это ограничение работы.